# 02. 실험 실행 창구 (Colab T4 / Kaggle 백업, ADR-0005)

역할: **브랜치 체크아웃 -> 데이터 준비 -> `scripts/run_experiment.py` 실행 -> 실행 기록 커밋/push**

- 로직은 모두 `src/sr_pipeline/`에 두며, 이 노트북에는 학습·평가 로직을 작성하지 않습니다.
- 실행 기록(`results/runs/<run_id>/`)은 **본인 이름 브랜치**에 커밋합니다. main에는 직접 push하지 않습니다(PR + 승인 1명).
- Colab과 Kaggle은 같은 명령을 쓰고 **경로 인자만** 다릅니다(셀 1이 환경을 감지해 채웁니다).
  - Colab: 공유 Drive의 압축 파일 1개를 런타임 로컬로 복사해 `--data-archive`로 풀어 씁니다. Drive 마운트가 안 되면 셀 1에서 `DATA_SOURCE = "gdown"`으로 바꾸고 셀 3에서 공유 링크로 받습니다.
  - Kaggle: 비공개 Dataset이 `/kaggle/input/...`에 붙으므로 `--data-dir`로 그 폴더를 그대로 씁니다.
- `configs/fair_comparison.yaml`의 `pilot_determined`가 `null`인 동안(T4.2 전)에는 본 실행이 시작 전에 거부됩니다. 이는 정상입니다.
- 각 셀은 명령이 실패하면 그 자리에서 멈춥니다. 다음 셀로 넘어가기 전에 출력을 확인하세요.
- **토큰은 셀 7에서 `getpass`로만 입력**합니다. 파일·출력·`.git/config`에 남지 않습니다. push가 실패하면 셀 7만 다시 실행하면 됩니다(셀 6을 다시 실행해도 같은 커밋을 돌려줍니다). 실행 후 이 노트북의 셀 출력은 지우고 커밋하세요.

In [ ]:
# 1. 실행 파라미터 (환경을 감지해 경로를 채운다)
import importlib.util, os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "YongMin"        # 본인 이름 브랜치 (YongMin / Gawoun). main은 push하지 않는다.
CONFIG_ID = "baseline"    # bicubic, baseline, A, B, AB, C, Bp (+ 승격 시 AC, BC, ABC)
SEED = 0                  # 0, 1, 2
RESUME_RUN_ID = ""        # 런타임이 끊긴 실행을 이어서 학습할 때만 그 run_id(코드 커밋이 같을 때만 가능)

IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or (
    Path("/kaggle/input").exists() and importlib.util.find_spec("google.colab") is None
)
if IN_KAGGLE:
    DATA_DIR = "/kaggle/input"                      # 붙인 Dataset 아래에서 patch_pool/을 찾는다(읽기 전용, 복사 없음). 패치 풀이 든 Dataset을 둘 이상 붙였다면 그 폴더로 좁힌다.
    DATA_ARCHIVE = None
    CKPT_DIR = "/kaggle/working/checkpoints"        # Kaggle에는 Drive가 없다. 필요하면 working 출력을 내려받는다.
    WORK = Path("/kaggle/working")
else:
    DATA_DIR = "/content/data"                      # 런타임 로컬 해제 위치
    DATA_ARCHIVE = "/content/sr_data.zip"           # Drive에서 복사한(또는 gdown으로 받은) 압축 파일
    CKPT_DIR = "/content/drive/MyDrive/vp_project/checkpoints"   # Drive를 못 쓰면 "/content/checkpoints" + 셀 6의 COMMIT_WEIGHTS=True
    WORK = Path("/content")
DRIVE_ARCHIVE = "/content/drive/MyDrive/vp_project/sr_data.zip"  # 공유 Drive의 압축 파일 위치(폴더 구성에 맞게 수정)
DATA_SOURCE = "drive"     # Colab: "drive"(Drive 마운트 후 복사) 또는 "gdown"(마운트가 안 될 때, 공유 링크로 받기 — ADR-0005 대체 경로)
ARCHIVE_SHA256 = ""       # 원본 zip의 sha256(64자리). 주면 해제 전에 확인한다(특히 gdown으로 받았을 때 권장).
REPO_DIR = WORK / "VideoProcessing-Project"
assert BRANCH != "main", "실행 기록은 이름 브랜치에 커밋합니다."


def run(*cmd, **kwargs):
    """명령을 실행하고, 실패하면 셀을 멈춘다(`!` 셸 매직은 실패해도 다음 줄로 넘어간다)."""
    subprocess.run([str(part) for part in cmd], check=True, **kwargs)


print("환경:", "Kaggle" if IN_KAGGLE else "Colab/로컬", "| 브랜치:", BRANCH, "| 구성:", CONFIG_ID, "| 시드:", SEED)

In [ ]:
# 2. 브랜치 가져오기 (다시 실행하면 BRANCH로 바꿔 최신으로 맞춘다)
if (REPO_DIR / ".git").is_dir():
    run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
    run("git", "-C", REPO_DIR, "checkout", BRANCH)
    run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
else:
    run("git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR)
current = subprocess.run(
    ["git", "-C", str(REPO_DIR), "branch", "--show-current"], capture_output=True, text=True, check=True
).stdout.strip()
assert current == BRANCH, f"현재 브랜치({current})가 BRANCH({BRANCH})와 다릅니다."
os.chdir(REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt")
run("git", "log", "--oneline", "-3")

In [ ]:
# 3. 데이터 준비
# Colab: 압축 파일 1개만 런타임 로컬로 가져온다(작은 파일 수만 개를 Drive에서 복사하면 매우 느리다). 해제는 셀 4의 실행 명령이 한다.
#   DATA_SOURCE = "drive": Drive를 마운트해 복사.   "gdown": 마운트 없이 공유 링크로 받기(ADR-0005 대체 경로; 링크는 getpass로만 입력, 출력·파일에 남기지 않음).
# Kaggle: 할 일 없음(Dataset이 /kaggle/input에 붙어 있어야 한다. 오른쪽 Input 패널의 Add Input).
import shutil
if not IN_KAGGLE:
    if DATA_SOURCE == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        shutil.copyfile(DRIVE_ARCHIVE, DATA_ARCHIVE)
    elif DATA_SOURCE == "gdown":
        import getpass
        run(sys.executable, "-m", "pip", "install", "-q", "gdown")
        import gdown
        link = getpass.getpass("Drive 공유 링크(zip 파일 하나, 화면에 표시되지 않음): ")
        try:
            gdown.download(url=link, output=DATA_ARCHIVE, fuzzy=True, quiet=True)
        except Exception:
            # gdown의 오류 메시지는 링크(파일 ID)를 담는다. 출력에 남지 않게 고정 안내로 바꾼다.
            raise RuntimeError("내려받기 실패: 링크 권한(링크가 있는 모든 사용자)과 파일 링크 여부를 확인하세요.") from None
        finally:
            del link
        assert Path(DATA_ARCHIVE).is_file(), "내려받기 실패: 링크 권한(링크가 있는 모든 사용자)과 파일 링크 여부를 확인하세요."
    else:
        raise ValueError(f"DATA_SOURCE는 'drive' 또는 'gdown'이어야 합니다: {DATA_SOURCE!r}")
    print(f"압축 파일 준비 완료: {Path(DATA_ARCHIVE).stat().st_size:,} B")

In [ ]:
# 4. 단일 실행 (학습 + 통합 평가 + 실행 기록 저장). Colab과 Kaggle이 같은 명령이고 경로 인자만 다르다.
cmd = [sys.executable, "scripts/run_experiment.py", "--config", CONFIG_ID, "--seed", str(SEED),
       "--data-dir", DATA_DIR, "--checkpoint-dir", CKPT_DIR]
if DATA_ARCHIVE:
    cmd += ["--data-archive", DATA_ARCHIVE]
    if ARCHIVE_SHA256:
        cmd += ["--archive-sha256", ARCHIVE_SHA256]
if RESUME_RUN_ID:
    cmd += ["--resume-run-id", RESUME_RUN_ID]
env = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src"), "PYTHONUNBUFFERED": "1"}
RUN_ID = None
with subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:   # 진행 출력을 그대로 보여 주면서 run_id 줄을 읽는다
        print(line, end="")
        if line.startswith("run_id: "):
            RUN_ID = line.split(":", 1)[1].strip()
if process.returncode != 0:
    raise RuntimeError(f"실행이 실패했습니다(종료 코드 {process.returncode}). 위 출력을 확인하세요.")
assert RUN_ID, "실행 출력에서 run_id를 찾지 못했습니다."

In [ ]:
# 5. 실행 기록 확인
RUN_DIR = REPO_DIR / "results" / "runs" / RUN_ID
assert RUN_DIR.is_dir(), f"실행 기록 폴더가 없습니다: results/runs/{RUN_ID}"
print("run_id:", RUN_ID)
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(RUN_DIR), f"{path.stat().st_size:,} B")
import pandas as pd
display(pd.read_csv(RUN_DIR / "summary.csv"))

In [ ]:
# 6. 실행 기록 커밋 (이름 브랜치). 커밋은 results/runs/<run_id>/ 만 담는다.
# 다시 실행해도 안전하다: 이미 커밋된 기록이면 새 커밋 없이 그 커밋을 돌려준다.
sys.path.insert(0, str(REPO_DIR / "src"))
from sr_pipeline.engine.publish import PushFailed, bundle_run_record, commit_run_record, push_branch

COMMIT_WEIGHTS = False   # Drive를 쓸 수 없는 경우에만 True: 평가 체크포인트(weights.pt, 약 150KB)를 실행 기록과 함께 커밋한다.
TRAINED = CONFIG_ID != "bicubic"   # Bicubic 하한은 학습하지 않아 train_log.json·체크포인트가 없다
GIT_NAME = input("커밋 작성자 이름 (이 커밋에만 사용): ").strip()
GIT_EMAIL = input("커밋 작성자 이메일 (이 커밋에만 사용): ").strip()
sha = commit_run_record(
    REPO_DIR, RUN_DIR, message=f"results: {CONFIG_ID} seed {SEED}", author_name=GIT_NAME, author_email=GIT_EMAIL,
    weights_path=Path(CKPT_DIR) / RUN_ID / "weights.pt" if COMMIT_WEIGHTS and TRAINED else None,
    trained=TRAINED,
)
print("커밋:", sha[:8])

In [ ]:
# 7. push (이름 브랜치). 토큰은 getpass로만 받고 어디에도 저장하지 않는다. 실패하면 이 셀만 다시 실행한다.
# - 토큰을 입력할 수 없거나 계속 실패하면 실행 기록 zip을 내려받아 로컬 VS Code에서 내 Git 자격 증명으로 커밋한다(ADR-0005).
import getpass
token = ""
make_bundle = False
try:
    GH_USER = input("GitHub 사용자명 (비워 두면 push 없이 zip 생성): ").strip()
    if GH_USER:
        token = getpass.getpass("GitHub 토큰 (비워 두면 push 없이 zip 생성, 저장되지 않음): ")
    if not GH_USER or not token:
        print("사용자명 또는 토큰이 비어 있어 push를 건너뛰고 실행 기록 zip을 만듭니다.")
        make_bundle = True
    else:
        push_branch(REPO_DIR, BRANCH, username=GH_USER, token=token)
        print("push 완료:", BRANCH)
except (PushFailed, ValueError, EOFError, OSError, RuntimeError) as error:
    detail = str(error).replace(token, "***") if token else str(error)
    print(detail)
    make_bundle = True
finally:
    token = ""
    del token
if make_bundle:
    bundle = bundle_run_record(RUN_DIR, WORK / f"{RUN_ID}.zip")
    print("실행 기록 zip을 내려받아 로컬 VS Code에서 내 Git 자격 증명으로 커밋하세요(ADR-0005):", bundle.name)
    try:
        from google.colab import files
        files.download(str(bundle))      # Colab: 브라우저/확장으로 바로 내려받는다. Kaggle은 Output 탭에서 받는다.
    except ImportError:
        pass